In [78]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression

In [105]:
df = sns.load_dataset('iris')
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [97]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded = ohe.fit_transform(df[['species']])
encoded
ohe.categories_

[array(['setosa', 'versicolor', 'virginica'], dtype=object)]

In [106]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns = ['species']), df['species'], test_size=0.2, random_state=0)

In [14]:
len(np.unique(y_test))

3

In [23]:
k = len(np.unique(y_test))
np.ones(shape = (k, X_train.shape[1]))

array([[1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.]])

In [115]:
class SoftmaxRegression:

    def __init__(self, epochs: int, learning_rate: float):
        self.epochs = epochs
        self.learning_rate = learning_rate
        self.coef_ = None
        self.intercept_ = None
        self.ohe = OneHotEncoder(sparse_output=False)
    
    def __softmax(self, Z: np.ndarray) -> np.ndarray:
        """
        Z: shape (n, k) — logits, one row per sample, one column per class
        Returns S: shape (n, k) — row-wise probabilities summing to 1
        """
        Z_shifted = Z - np.max(Z, axis=1, keepdims=True)
        exp_Z = np.exp(Z_shifted)
        S = exp_Z / np.sum(exp_Z, axis=1, keepdims=True)
        return S

    def fit(self, X: np.ndarray | pd.DataFrame, y: np.ndarray):
        """
        X: np.ndarray(n_samples, n_features)\n
        y: np.ndarray(n_samples,)\n
        Fits the Data into model and Calculate Coefficients and Intercepts
        """
        X = np.asarray(X)
        X = np.insert(X, 0, 1, axis = 1)
        k = len(np.unique(y))
        weights = np.ones(shape = (k, X.shape[1]))
             
        y_ohe = self.ohe.fit_transform(y.to_frame(name = 'target'))

        for i in range(self.epochs):
            y_pred = self.__softmax(np.dot(X, weights.T))
            weights += self.learning_rate* np.dot((y_ohe - y_pred).T, X)/X.shape[0]
        
        self.coef_ = weights[:, 1:]
        self.intercept_ = weights[:, 0]

    def predict_proba(self, X:np.ndarray | pd.DataFrame) -> np.ndarray:
        X = np.asarray(X)
        probs = self.__softmax(np.dot(np.insert(X, 0, 1, axis = 1), np.insert(self.coef_, 0, self.intercept_, axis=1).T))
        pred = probs.argmax(axis=1)
        return probs

    def predict(self, X:np.ndarray | pd.DataFrame) -> np.ndarray:
        X = np.asarray(X)
        probs = self.__softmax(np.dot(np.insert(X, 0, 1, axis = 1), np.insert(self.coef_, 0, self.intercept_, axis=1).T))
        pred = probs.argmax(axis=1)
        labels = self.ohe.categories_[0][pred]
        return labels

In [116]:
clf = SoftmaxRegression(epochs = 500, learning_rate=0.1)

clf.fit(X_train, y_train)

clf.predict_proba(X_test)

array([[7.62042227e-05, 3.20638022e-02, 9.67859994e-01],
       [1.27524055e-02, 9.03087146e-01, 8.41604487e-02],
       [9.95004523e-01, 4.99543299e-03, 4.37066603e-08],
       [3.97487092e-05, 1.29531317e-01, 8.70428934e-01],
       [9.71508143e-01, 2.84900808e-02, 1.77623560e-06],
       [1.13004475e-05, 1.32501246e-02, 9.86738575e-01],
       [9.84095620e-01, 1.59036532e-02, 7.26370268e-07],
       [1.25534984e-02, 8.33655744e-01, 1.53790757e-01],
       [6.00968079e-03, 8.22273518e-01, 1.71716801e-01],
       [3.48821289e-02, 8.87148487e-01, 7.79693841e-02],
       [1.18668339e-04, 1.54424765e-01, 8.45456567e-01],
       [2.22353111e-02, 8.30915837e-01, 1.46848852e-01],
       [6.38615012e-03, 7.42524516e-01, 2.51089334e-01],
       [8.09701499e-03, 7.70440982e-01, 2.21462003e-01],
       [6.42799051e-03, 6.60200678e-01, 3.33371331e-01],
       [9.84141978e-01, 1.58574374e-02, 5.84860217e-07],
       [9.61895838e-03, 6.64448926e-01, 3.25932115e-01],
       [6.51018935e-03, 6.20594

In [117]:
clf.predict(X_test)

array(['virginica', 'versicolor', 'setosa', 'virginica', 'setosa',
       'virginica', 'setosa', 'versicolor', 'versicolor', 'versicolor',
       'virginica', 'versicolor', 'versicolor', 'versicolor',
       'versicolor', 'setosa', 'versicolor', 'versicolor', 'setosa',
       'setosa', 'virginica', 'versicolor', 'setosa', 'setosa',
       'virginica', 'setosa', 'setosa', 'versicolor', 'versicolor',
       'setosa'], dtype=object)

In [119]:
y_test.values == clf.predict(X_test)

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True])